In [0]:
from pyspark.sql import functions as F

CATALOG = "olist"
LANDING = f"/Volumes/{CATALOG}/raw/landing"
CHECKPOINTS = f"/Volumes/{CATALOG}/raw/landing/_checkpoints"
TABLES = ["orders", "order_items", "payments", "customers", "reviews",
          "products", "sellers", "category_translation", "geolocation"]

def load_bronze(table: str) -> None:
    files = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINTS}/{table}/schema")
        .load(f"{LANDING}/{table}")
    )
    rows = (
        files
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_business_date",
                    F.regexp_extract("_source_file", r"business_date=(\d{4}-\d{2}-\d{2})", 1).cast("date"))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    (
        rows.writeStream
        .option("checkpointLocation", f"{CHECKPOINTS}/{table}/checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.bronze.{table}")
        .awaitTermination()
    )
    print(f"{table}: done")

for t in TABLES:
    load_bronze(t)

orders: done
order_items: done
payments: done
customers: done
reviews: done
products: done
sellers: done
category_translation: done
geolocation: done


In [0]:
%sql
SELECT _business_date, COUNT(*) AS rows
FROM olist.bronze.orders
GROUP BY _business_date
ORDER BY _business_date

_business_date,rows
2017-01-05,32
2017-01-06,7
2017-01-07,33
2017-01-08,6
2017-01-09,12
2017-01-10,12
2017-01-11,47
2017-01-12,27
2017-01-13,28
2017-01-14,20
